<a href="https://colab.research.google.com/github/zero0r0/GetSongSetListTool/blob/main/SONG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 2. ついでに軽量なJavaScriptエンジン「Deno」もインストール（yt-dlpが推奨しているもの）
!curl -fsSL https://deno.land/x/install/install.sh | sh
import os
os.environ['PATH'] += ":/root/.deno/bin"

# 3. ライブラリを最新に
!pip install -U google-genai yt-dlp

In [ ]:
import os
import time
import math
import subprocess
import yt_dlp
from google import genai
from google.colab import userdata, files

def get_flash_model():
    """利用可能なFlashモデルを自動選択"""
    available_models = [m.name for m in client.models.list()]
    # 今回は一旦雑に手書きにする（本来は入力待ちにして、modelを選択させるのが良い気がする）
    for candidate in ["models/gemini-2.5-flash", "models/gemini-2.0-flash", "models/gemini-2.0-flash-001", "gemini-1.5-flash"]:
        if candidate in available_models: return candidate
    return [m for m in available_models if "flash" in m.lower()][0]

def download_full_audio(url, output_name="full_audio"):
    """動画全体を一度だけダウンロードする"""
    print(f"📥 動画全体の音声をダウンロード中... (これには少し時間がかかります)")
    output_filename = f"{output_name}.m4a"

    if os.path.exists(output_filename):
        os.remove(output_filename)

    # クッキーファイルがあれば使用する設定
    cookie_file = "cookies.txt"

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': output_name,
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'm4a'}],
        'quiet': True,
        'no_warnings': True,
        # ボット検知回避のための設定を追加
        'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'referer': 'https://www.google.com/',
        #'extractor_args': {'youtube': {'player_client': ['web_safari']}},
    }

    if os.path.exists(cookie_file):
        print(f"🍪 {cookie_file} を使用して認証を試みます")
        ydl_opts['cookiefile'] = cookie_file

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        print("✅ ダウンロード完了！")
        return output_filename
    except Exception as e:
        print(f"❌ ダウンロードエラー: {e}")
        print("💡 ヒント: cookies.txtをアップロードすると解決する場合があります。")
        return None

def split_audio_local(input_file, start_time, end_time, output_file):
    command = [
        'ffmpeg', '-y',
        '-ss', str(start_time),
        '-to', str(end_time),
        '-i', input_file,
        '-c', 'copy',
        '-loglevel', 'error',
        output_file
    ]
    subprocess.run(command, check=True)
    return output_file

def analyze_part(file_path, base_offset_seconds):
    target_model = get_flash_model()
    print(f"🤖 モデル: {target_model}")
    print(f"      📤 AIに送信中...", end="")
    uploaded_file = client.files.upload(file=file_path)

    while uploaded_file.state.name == "PROCESSING":
        time.sleep(2)
        uploaded_file = client.files.get(name=uploaded_file.name)
    print(" [完了] → 🧠 解析中...")

    offset_str = str(time.strftime('%H:%M:%S', time.gmtime(base_offset_seconds)))
    prompt = f"""
    この音声ファイルを解析して、歌枠のセットリストを作成してください。
    【ルール】
    1. 各曲の開始時間を【00:00】曲名 / アーティスト名の形式で示してください。
    2. この音声は動画の開始から {offset_str} 経過した時点からのものです。
       出力するタイムスタンプは、この時間を加算して動画全体での正しい時間を出力してください。
    3. 曲名 / アーティスト名 で表記してください
    4. 歌っていないお喋り部分は除外してください。
    5. 曲が見つからない場合は特になにも出力しなくてよいです。
    """

    try:
        response = client.models.generate_content(model=target_model, contents=[prompt, uploaded_file])
        return response.text
    finally:
        try: client.files.delete(name=uploaded_file.name)
        except: pass

def main():
    url = input("YouTube URL: ")

    # 情報取得時にもクッキーを考慮
    info_opts = {'quiet': True}
    if os.path.exists("cookies.txt"): info_opts['cookiefile'] = "cookies.txt"

    try:
        with yt_dlp.YoutubeDL(info_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            duration = info['duration']
            title = info['title']
    except Exception as e:
        print(f"❌ 情報取得エラー: {e}")
        return

    print(f"\n🎬 {title} (総時間: {math.ceil(duration/60)}分)")
    full_audio_path = download_full_audio(url)
    if not full_audio_path: return

    results = []
    interval = 1800
    num_segments = math.ceil(duration / interval)

    print("\n🚀 ローカル分割解析を開始します...")

    for i in range(num_segments):
        start = i * interval
        end = min((i + 1) * interval, duration)
        print(f"\n📂 [{i+1}/{num_segments}] セグメント ({math.ceil(start/60)}分〜{math.ceil(end/60)}分)")
        segment_file = f"part_{i}.m4a"
        split_audio_local(full_audio_path, start, end, segment_file)

        try:
            res_text = analyze_part(segment_file, start)
            print(f"📝 結果:\n{res_text}\n" + "-"*30)
            results.append(res_text)
        except Exception as e:
            print(f"⚠️ エラー: {e}")
        finally:
            if os.path.exists(segment_file): os.remove(segment_file)

    if os.path.exists(full_audio_path): os.remove(full_audio_path)

    final_output = "\n\n".join(results)
    with open("setlist.txt", "w", encoding="utf-8") as f:
        f.write(f"TITLE: {title}\nURL: {url}\n\n{final_output}")
    files.download("setlist.txt")


# --- クライアント設定 ---
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
# TODO: モデルをプルダウンとかで選択させたいかも...
target_model = get_flash_model()
print(f"🤖 モデル: {target_model}")

if __name__ == "__main__":
    main()

